# Explainability Lab (CYBERUS Master in Cybersecurity)
**Duration:** 1h30 (90 minutes)

**How this lab works:**
- All library boilerplate (`SHAP`, `LIME`, `Captum`, `Zennit`, `Quantus`, `DiCE`, and `fooling_lime_shap`) is already implemented and tuned to run quickly on CPU (~60 seconds total).
- **Your job** is to execute each section, inspect the generated plots and tables, identify subtle methodological or security flaws in the pipeline, and answer the **[STUDENT ANSWER]** blocks in 2–4 concise sentences citing the exact numbers/features you observe.
- Answering the questions is optional but if you want to do it, please **do not use AI** to answer as these are understanding questions.

**Sections:**
- **Part 0:** Setup, Data Preprocessing, Random Forest & PyTorch MLP
- **Part 1:** Perturbation Explanations (`SHAP` vs. `LIME`)
- **Part 2:** Gradient-Based Attributions (`Captum` & `Zennit` LRP) & `Quantus` Evaluation
- **Part 3:** Counterfactual Explanations & Actionability (`DiCE`)
- **Part 4:** Adversarial Attacks on Post-Hoc Explanations (`Fooling LIME & SHAP`)
- **Part 5 (Optional / GPU Demo):** LLM Token Attribution (`Captum` & `LXT`)
- **Part 6 (Optional / GPU Demo):** VLM Token/Pixel Attribution (`TAM`)

# Part 0: Setup, Preprocessing & Models

In [ ]:
import os
import random
import warnings
from typing import List, Callable, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from captum.attr import (
    Saliency,
    IntegratedGradients,
    NoiseTunnel,
    InputXGradient,
    LRP,
)
from captum.attr._utils.attribution import GradientAttribution
from captum.metrics import sensitivity_max, infidelity

from zennit.attribution import Gradient
from zennit.composites import EpsilonPlus

import quantus

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# %% [markdown]
# ## Load & Preprocess the UCI Adult Dataset

# %%
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame.copy()
df = df.replace("?", np.nan).dropna()
df["class"] = (df["class"] == ">50K").astype(int)

target_col = "class"
X_df = df.drop(columns=[target_col])
y = df[target_col].values

cat_cols = X_df.select_dtypes(include=["category", "object"]).columns.tolist()
num_cols = [c for c in X_df.columns if c not in cat_cols]

# Split BEFORE fitting preprocessors to prevent train-test data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=SEED, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ]
)

X_train = preprocess.fit_transform(X_train_raw)
X_test = preprocess.transform(X_test_raw)

feature_names_num = num_cols
feature_names_cat = list(preprocess.named_transformers_["cat"].get_feature_names_out(cat_cols))
feature_names_all = feature_names_num + feature_names_cat
cat_indices = [feature_names_all.index(c) for c in feature_names_cat]

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)


## Train Random Forest & PyTorch MLP

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_pred_rf = rf.predict(X_test)
print("=== Random Forest Test Performance ===")
print(classification_report(y_test, y_pred_rf, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 3))

In [ ]:
class MLPModel(nn.Module):
    """
    Fully-connected MLP for tabular classification.
    NOTE: Look closely at `forward()` and how `criterion` is defined below!
    """
    def __init__(self, n_layers: int, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers += [nn.Linear(hidden_dim, output_dim)]
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.network(x)
        #x = F.softmax(x, dim=1)  # <-- AUDIT THIS LINE FOR QUESTION 0.1
        return x


def train_mlp(model: nn.Module, loader: DataLoader, epochs: int, criterion: nn.Module,
              optimizer: optim.Optimizer, device: torch.device) -> None:
    model.train()
    for epoch in range(epochs):
        total_loss, n_batches = 0.0, 0
        for x_b, y_b in loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            out = model(x_b)
            loss = criterion(out, y_b)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs:02d} | loss={total_loss/n_batches:.4f}")


def predict_mlp(model: nn.Module, loader: DataLoader, device: torch.device) -> np.ndarray:
    model.eval()
    preds = []
    with torch.no_grad():
        for x_b, _ in loader:
            probs = model(x_b.to(device))
            preds.append(probs.cpu().numpy())
    return np.argmax(np.concatenate(preds, axis=0), axis=1)




In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mlp = MLPModel(n_layers=3, input_dim=X_train.shape[1], hidden_dim=64, output_dim=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=1e-3)

print("\n=== Training PyTorch MLP ===")
train_mlp(mlp, train_loader, epochs=15, criterion=criterion, optimizer=optimizer, device=device)

pred_mlp = predict_mlp(mlp, test_loader, device)
y_pred_t = torch.from_numpy(pred_mlp).to(device)
print(f"MLP Test Accuracy: {(pred_mlp == y_test).mean() * 100:.2f}%")

### Code Audit Question 0.1 (Before moving to Part 2)
Inspect `MLPModel.forward()` and `criterion = nn.CrossEntropyLoss()`.
1. Why is applying `F.softmax(x, dim=1)` inside `forward()` combined with `nn.CrossEntropyLoss()` a bug during training?
2. How does keeping `F.softmax` at the output affect gradient-based XAI methods (`Saliency`, `IntegratedGradients`, `LRP`) in Part 2 compared to explaining pre-softmax logits?

**[STUDENT ANSWER 0.1]:**
> *Write your 2–3 sentence answer here.*

<details>
<summary><b>Click to reveal Answer 0.1</b></summary>

1. **Training bug:** `nn.CrossEntropyLoss()` already applies `LogSoftmax` internally, so keeping `F.softmax()` in `forward()` applies softmax twice and squashes gradients. To fix the `MLPModel.forward()` comment the `x = F.softmax(x, dim=1)` line and retrain the model for the rest of the practical.
2. **XAI impact:** Taking gradients (`Saliency`, `IntegratedGradients`, `LRP`) after `F.softmax` causes gradient saturation on confident predictions and couples the target class gradient to all other class logits.

</details>

# Part 1: Perturbation-Based Explanations (`SHAP` vs. `LIME`)

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer
from IPython.display import display, HTML

# Fast subset for interactive classroom runtime
N_EXPLAIN = 100
X_sub = X_test[:N_EXPLAIN]
pred_sub = y_pred_rf[:N_EXPLAIN]


## SHAP (TreeSHAP)

In [ ]:

# 1. TreeSHAP on Random Forest
explainer_shap = shap.TreeExplainer(rf, feature_names=feature_names_all)
shap_values = explainer_shap(X_sub)

print("Random Forest predicted class for test sample 0:", y_pred_rf[0],
      "| P(>50K):", round(y_prob_rf[0], 3))

# Waterfall plot for sample 0 (class 1: >50K)
shap.plots.waterfall(shap_values[0, :, 1])

In [ ]:
# Global summary plot across 100 test instances
shap.plots.beeswarm(shap_values[:, :, 1])

In [ ]:
# Bar plot
shap.plots.bar(shap_values[:, :, 0])

## LIME

In [ ]:
# 2. LIME on the exact same instance (sample 0)
# NOTE: Look closely at `categorical_features` and `discretize_continuous` below!
explainer_lime_buggy = LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names_all,
    class_names=["<=50K", ">50K"],
    categorical_features=cat_indices,
    discretize_continuous=True,
    random_state=SEED,
)

exp_lime_0 = explainer_lime_buggy.explain_instance(
    data_row=X_test[0],
    predict_fn=rf.predict_proba,
    num_features=8,
)
display(HTML(exp_lime_0.as_html()))


In [ ]:
# Stability check: re-run LIME with a different random seed on the SAME sample
explainer_lime_seed2 = LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names_all,
    class_names=["<=50K", ">50K"],
    categorical_features=cat_indices,
    discretize_continuous=True,
    random_state=999,
)
exp_lime_0_b = explainer_lime_seed2.explain_instance(
    data_row=X_test[0],
    predict_fn=rf.predict_proba,
    num_features=8,
)

display(HTML(exp_lime_0.as_html()))


In [ ]:
print("LIME Run 1 (seed=42) top 3:", exp_lime_0.as_list()[:3])
print("LIME Run 2 (seed=999) top 3:", exp_lime_0_b.as_list()[:3])

### Comprehension Questions — Part 1
1. **SHAP vs. LIME Disagreement:** Compare the top 3 features explaining `X_test[0]` in the SHAP waterfall plot versus LIME Run 1. Why can two post-hoc explainers disagree on feature rankings for the exact same model and input?
2. **Instability & Encoding Risk:** Look at `LIME Run 1` vs. `LIME Run 2` (different seeds), and consider how `OrdinalEncoder` encodes categorical columns like `education` or `occupation` in Part 0. Why is `OrdinalEncoder` + Gaussian perturbation problematic for tabular LIME?


# Part 2: Gradient-Based Explanations & `Quantus` Evaluation

In [ ]:
# Visualization helpers

def visualize_importances(feature_names: List[str], importances: np.ndarray,
                          title: str = "Average Feature Importances") -> None:
    # Find the indices of the top 3 highest importances
    top_3_indices = set(np.argsort(importances)[-3:])
    
    # Highlight top 3 in orange/coral, keep the rest in muted slate gray
    colors = ['#ff7f0e' if i in top_3_indices else '#8c969e' for i in range(len(importances))]
    
    y_pos = np.arange(len(feature_names))
    
    # Adjusted figsize to give vertical room for horizontal bars
    plt.figure(figsize=(10, max(4, len(feature_names) * 0.4)))
    plt.barh(y_pos, importances, align="center", color=colors)
    plt.yticks(y_pos, feature_names)
    
    # Invert y-axis so the first feature appears at the top
    plt.gca().invert_yaxis()
    
    plt.xlabel("Importance")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def normalize_attr(attr: np.ndarray) -> np.ndarray:
    eps = 1e-16
    denom = np.max(np.abs(attr), axis=0) + eps
    return attr / denom

## Compute Attributions Across 5 Methods (`N = 100` test samples for fast CPU execution)

In [ ]:
N_GRAD = 100
X_grad_t = X_test_t[:N_GRAD].to(device)
y_grad_t = y_pred_t[:N_GRAD].to(device)
X_grad_np = X_test[:N_GRAD]
y_grad_np = pred_mlp[:N_GRAD]

### Saliency

In [ ]:
# 1. Saliency
xai_saliency = Saliency(mlp)
attr_sal = xai_saliency.attribute(X_grad_t.clone().requires_grad_(True), target=y_grad_t, abs=False).detach().cpu().numpy()

In [ ]:
visualize_importances(feature_names_all, np.abs(attr_sal.mean(axis=0)), title="Saliency Feature Importances")

### SmoothGrad

In [ ]:
# 2. SmoothGrad (NoiseTunnel over Saliency)
xai_sg = NoiseTunnel(xai_saliency)
attr_sg = xai_sg.attribute(X_grad_t.clone().requires_grad_(True), nt_type="smoothgrad",
                           nt_samples=15, stdevs=0.1, target=y_grad_t, abs=False).detach().cpu().numpy()

In [ ]:
#TODO: Visualize SmoothGrad importances

### InputxGradient

In [ ]:
# 3. Input x Gradient
xai_ixg = InputXGradient(mlp)
attr_ixg = xai_ixg.attribute(X_grad_t.clone().requires_grad_(True), target=y_grad_t).detach().cpu().numpy()

In [ ]:
#TODO: Visualize Input x Gradient importances

### Integrated Gradients (Mean Baseline)

In [ ]:
# 4a. Integrated Gradients (Mean Baseline)
xai_ig = IntegratedGradients(mlp)
mean_baseline = X_train_t.mean(dim=0, keepdim=True).repeat(N_GRAD, 1).to(device)
attr_ig_mean = xai_ig.attribute(X_grad_t, baselines=mean_baseline, target=y_grad_t, n_steps=30).detach().cpu().numpy()

In [ ]:
#TODO: Visualize Integrated Gradients importances

### Integrated Gradients (Zero Baseline)

In [ ]:
# 4b. Integrated Gradients (Zero Baseline — spot the limitation!)
zero_baseline = torch.zeros_like(X_grad_t)
attr_ig_zero = xai_ig.attribute(X_grad_t, baselines=zero_baseline, target=y_grad_t, n_steps=30).detach().cpu().numpy()


In [ ]:
#TODO: Visualize Integrated Gradients importances

### LRP (Zennit)

In [ ]:
# 5. LRP (Zennit EpsilonPlus)
composite = EpsilonPlus()
xai_lrp = Gradient(mlp, composite)
targets_oh = F.one_hot(y_grad_t, num_classes=2).float()
with xai_lrp:
    _, attr_lrp_t = xai_lrp(X_grad_t.clone().requires_grad_(True), targets_oh)
attr_lrp = attr_lrp_t.detach().cpu().numpy()

In [ ]:
#TODO: Visualize LRP importances

### LRP (Captum)

In [ ]:
# 6 LRP (Captum implementation)
lrp = LRP(mlp)
attr_lrp_captum = lrp.attribute(X_grad_t.clone().requires_grad_(True), target=y_grad_t).detach().cpu().numpy()


In [ ]:
#TODO: Visualize LRP importances

### Comparison

In [ ]:
# Compare global mean normalized importances across methods
summary_df = pd.DataFrame({
    "Feature": feature_names_all,
    "Saliency": np.mean(np.abs(normalize_attr(attr_sal)), axis=0),
    "SmoothGrad": np.mean(np.abs(normalize_attr(attr_sg)), axis=0),
    "InputXGrad": np.mean(np.abs(normalize_attr(attr_ixg)), axis=0),
    "IG (Mean Base)": np.mean(np.abs(normalize_attr(attr_ig_mean)), axis=0),
    "IG (Zero Base)": np.mean(np.abs(normalize_attr(attr_ig_zero)), axis=0),
    "LRP (Zennit)": np.mean(np.abs(normalize_attr(attr_lrp)), axis=0),
    "LRP (Captum)": np.mean(np.abs(normalize_attr(attr_lrp_captum)), axis=0),
}).set_index("Feature")

print("=== Mean Normalized |Attribution| Across 100 Test Samples ===")
display(summary_df.round(4))

In [ ]:
X_grad_np.shape

## Quantitative Evaluation (`Quantus` & `Captum` Faithfulness / Robustness)

In [ ]:
def make_wrapper(method_name: str):
    def wrapper(model, inputs, targets, **kwargs):
        x_t = torch.tensor(inputs, dtype=torch.float32, device=device, requires_grad=True)
        y_t = torch.tensor(targets, dtype=torch.long, device=device)
        if method_name == "Saliency":
            out = xai_saliency.attribute(x_t, target=y_t, abs=False)
        elif method_name == "SmoothGrad":
            out = xai_sg.attribute(x_t, nt_type="smoothgrad", nt_samples=10, stdevs=0.1, target=y_t, abs=False)
        elif method_name == "InputXGrad":
            out = xai_ixg.attribute(x_t, target=y_t)
        elif method_name == "IG_Mean":
            base = X_train_t.mean(dim=0, keepdim=True).repeat(len(inputs), 1).to(device)
            out = xai_ig.attribute(x_t, baselines=base, target=y_t, n_steps=20)
        elif method_name == "IG_Zero":
            base = torch.zeros_like(x_t)
            out = xai_ig.attribute(x_t, baselines=base, target=y_t, n_steps=20)
        elif method_name == "LRP":
            y_oh = F.one_hot(y_t, num_classes=2).float()
            with xai_lrp:
                _, out = xai_lrp(x_t, y_oh)
        elif method_name == "LRP_Captum":
            out = lrp.attribute(x_t, target=y_t)
        return out.detach().cpu().numpy()
    return wrapper


metrics_quantus = {
    "RIS (Stability, lower=better)": quantus.RelativeInputStability(nr_samples=5, return_aggregate=True),
    "Faithfulness (higher=better)": quantus.FaithfulnessEstimate(
        abs=False, normalise=False, features_in_step=1, perturb_baseline="mean", return_aggregate=True
    ),
}

wrappers = {
    "Saliency": make_wrapper("Saliency"),
    "SmoothGrad": make_wrapper("SmoothGrad"),
    "InputXGrad": make_wrapper("InputXGrad"),
    "IG_Mean": make_wrapper("IG_Mean"),
    "IG_Zero": make_wrapper("IG_Zero"),
    "LRP": make_wrapper("LRP"),
    "LRP_Captum": make_wrapper("LRP_Captum"),
}

# Run on 40 samples so all 5 methods finish in ~10 seconds
q_res = quantus.evaluate(
    metrics=metrics_quantus,
    xai_methods=wrappers,
    model=mlp.eval(),
    x_batch=X_grad_np,
    y_batch=y_grad_np,
    explain_func_kwargs={},
    call_kwargs={"run": {"device": device}},
    verbose=False,
)

# # Add Captum Sensitivity & Infidelity
# def perturb_fn(inputs):
#     noise = torch.randn_like(inputs) * 0.01
#     return noise, inputs - noise

feature_stds = X_train_t.std(dim=0, keepdim=True)

def tabular_gaussian_perturb(inputs: torch.Tensor):
    # Scale noise by each feature's std (e.g., 10% of feature std)
    noise_scale = 0.1 * feature_stds.to(inputs.device)
    
    # Perturbation I ~ N(0, noise_scale^2)
    perturbation_I = torch.randn_like(inputs) * noise_scale
    
    # x - I
    perturbed_inputs = inputs - perturbation_I
    return perturbation_I, perturbed_inputs

# Precompute baseline (e.g., column means or zeros): shape (1, num_features)
baseline = X_train_t.mean(dim=0, keepdim=True)

def tabular_masking_perturb(inputs: torch.Tensor, drop_prob: float = 0.2):
    # Binary mask: 1 means "perturb/drop this feature", 0 means "keep"
    mask = (torch.rand_like(inputs) < drop_prob).float()
    
    base = baseline.to(inputs.device)
    
    # Since perturbed_inputs = inputs - perturbation_I, 
    # setting perturbation_I = mask * (inputs - baseline) ensures:
    # - where mask == 0: perturbed_inputs = inputs
    # - where mask == 1: perturbed_inputs = baseline
    perturbation_I = mask * (inputs - base)
    perturbed_inputs = inputs - perturbation_I
    
    return perturbation_I, perturbed_inputs



In [ ]:
eval_rows = []
captum_objs = {
    "Saliency": (xai_saliency, attr_sal),
    "SmoothGrad": (xai_sg, attr_sg),
    "InputXGrad": (xai_ixg, attr_ixg),
    "IG_Mean": (xai_ig, attr_ig_mean),
    "IG_Zero": (xai_ig, attr_ig_zero),
    "LRP_Captum": (lrp, attr_lrp_captum),
}

for name in wrappers:
    row = {
        "Method": name,
        # "Quantus_RIS (↓)": np.nanmean(q_res[name]["RIS (Stability, lower=better)"]),
        "Quantus_Faithfulness (↑)": np.nanmean(q_res[name]["Faithfulness (higher=better)"]),
    }
    if name in captum_objs:
        obj, at = captum_objs[name]
        sens = sensitivity_max(obj.attribute, X_grad_t, target=y_grad_t).mean().item()
        infid = infidelity(mlp,
                           tabular_gaussian_perturb, # Change the perturbation function to the tabular masking perturbation
                           X_grad_t,
                           torch.tensor(at, device=device),
                           target=y_grad_t).mean().item()
        row["Captum_Sensitivity (↓)"] = sens
        row["Captum_Infidelity (↓)"] = infid
    eval_rows.append(row)

display(pd.DataFrame(eval_rows).set_index("Method").round(4))

### Comprehension Questions — Part 2
1. **Baseline Trap (`IG Mean` vs. `IG Zero`):** Look at `summary_df` for `IG (Mean Base)` vs. `IG (Zero Base)`. Mathematically, why does Integrated Gradients with `zero_baseline` assign **exact zero attribution** whenever a feature's input value is `0.0` (e.g., a standardized continuous feature at the dataset mean, or the first category `0` of an `OrdinalEncoder`)?
2. **Robustness vs. Faithfulness Trade-off:** Using the evaluation table above, compare all XAI methods using on `Captum_Sensitivity` (robustness) and `Captum_Infidelity` (faithfulness). Which method is more stable under small input noise, and why? Which more is the most faithful ?
3. **Perturbation function**: Change the perturbation function from `tabular_gaussian_perturb` to `tabular_masking_perturb` and redo question 2. Can you explain why one perturbation function is better than another ?


**[STUDENT ANSWER 2.1]:**
> *Write your 2–3 sentence answer here.*

**[STUDENT ANSWER 2.2]:**
> *Write your 2–3 sentence answer here.*

**[STUDENT ANSWER 2.3]:**
> *Write your 2–3 sentence answer here.*

# Part 3: Counterfactual Explanations & Actionability (`DiCE`)

In [ ]:
import dice_ml

# Re-use clean raw DataFrames from Part 0 (no leakage)
train_dice_df = X_train_raw.copy()
train_dice_df["class"] = y_train

clf_dice = Pipeline(steps=[
    ("preprocessor", preprocess),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
])
clf_dice.fit(X_train_raw, y_train)

d_dice = dice_ml.Data(dataframe=train_dice_df, continuous_features=num_cols, outcome_name="class")
m_dice = dice_ml.Model(model=clf_dice, backend="sklearn")
exp_dice = dice_ml.Dice(d_dice, m_dice, method="random")


In [ ]:
query_instance = X_test_raw.iloc[0:1]
print("Original prediction:", clf_dice.predict(query_instance)[0])

# 1. Unconstrained counterfactuals (all features allowed to change)
cf_unconstrained = exp_dice.generate_counterfactuals(query_instance, total_CFs=3, desired_class="opposite")
print("\n--- Unconstrained Counterfactuals (Only Changes Shown) ---")
cf_unconstrained.visualize_as_dataframe(show_only_changes=True)


In [ ]:
# 2. Actionable counterfactuals (restrict immutable features like sex, race, native-country)
actionable_features = ["age", "education", "hours-per-week", "capital-gain", "occupation"]
cf_actionable = exp_dice.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
    features_to_vary=actionable_features,
    permitted_range={"age": [int(query_instance["age"].iloc[0]), 80]},  # age can only increase!
)
print("\n--- Actionable Counterfactuals (Immutable Features Locked) ---")
cf_actionable.visualize_as_dataframe(show_only_changes=True)

In [ ]:
# Quantitative Comparison on 30 Test Instances: Unconstrained vs. Actionable
N_CF = 100
q_batch = X_test_raw.iloc[:N_CF]

def evaluate_dice_batch(dice_exp, queries, **kwargs) -> Dict[str, float]:
    res = dice_exp.generate_counterfactuals(queries, total_CFs=1, desired_class="opposite", **kwargs)
    valid_flags, l1_dists, sparsities = [], [], []
    orig_preds = clf_dice.predict(queries)

    for i, cf_ex in enumerate(res.cf_examples_list):
        if cf_ex.final_cfs_df is None or len(cf_ex.final_cfs_df) == 0:
            valid_flags.append(0.0)
            continue
        cf_row = cf_ex.final_cfs_df.drop(columns=["class"]).iloc[0:1]
        orig_row = queries.iloc[i:i+1]

        # 1. Validity: did predicted class actually flip?
        cf_pred = clf_dice.predict(cf_row)[0]
        valid_flags.append(float(cf_pred != orig_preds[i]))

        # 2. Proximity: L1 distance on standardized numerical features
        orig_num = preprocess.named_transformers_["num"].transform(orig_row[num_cols])[0]
        cf_num = preprocess.named_transformers_["num"].transform(cf_row[num_cols])[0]
        l1_dists.append(np.sum(np.abs(cf_num - orig_num)))

        # 3. Sparsity: total number of features modified (num + cat)
        sparsities.append(int((cf_row.values[0] != orig_row.values[0]).sum()))

    return {
        "Validity (↑)": np.mean(valid_flags),
        "Mean L1 Proximity (↓)": np.mean(l1_dists) if l1_dists else np.nan,
        "Mean Sparsity / # Changed Features (↓)": np.mean(sparsities) if sparsities else np.nan,
    }



In [ ]:
metrics_unconstrained = evaluate_dice_batch(exp_dice, q_batch)
metrics_actionable = evaluate_dice_batch(exp_dice, q_batch, features_to_vary=actionable_features)

display(pd.DataFrame([metrics_unconstrained, metrics_actionable],
                     index=["Unconstrained DiCE", "Actionable DiCE"]).round(3))

### Comprehension Questions — Part 3
1. **Actionability Audit:** Inspect the `Unconstrained Counterfactuals` table for `query_instance`. Which feature edits produced by DiCE are **unrealistic or non-actionable** in practice (e.g., immutable attributes or decreasing `age`)?
2. **Cost of Recourse & Adversarial Evasion:** Compare `Validity` and `Mean L1 Proximity` between `Unconstrained DiCE` and `Actionable DiCE` across the 30 samples. Why does restricting `features_to_vary` increase the distance required to cross the decision boundary—and how does this relate to an attacker crafting a constrained evasion sample?
3. **Try**: You can try generating counterfactual explanations for other examples using the same code. Change the `actionable_features` or the `permitted_range` features.

**[STUDENT ANSWER 3.1]:**
> *Write your 2–3 sentence answer here.*

**[STUDENT ANSWER 3.2]:**
> *Write your 2–3 sentence answer here.*

**[STUDENT ANSWER 3.3]:**
> *Write your 2–3 sentence answer here.*

# Part 4: Fooling LIME & SHAP (Adversarial Attacks on Explanations)
Based on Slack et al. (2020), *Fooling LIME and SHAP: Adversarial Attacks on Post hoc Explanation Methods*.

**Threat Model:** A malicious model developer deploys a classifier `adv_lime` / `adv_shap` that discriminates strictly on `sex` on real in-distribution data ($f(x)$), but detects out-of-distribution perturbation probes generated by LIME/SHAP and switches to an innocuous proxy model $\psi(x)$ relying on `unrelated_column`.


## Fooling LIME

In [ ]:
from attacks.fooling_lime_shap import Adversarial_Lime_Model, Adversarial_Kernel_SHAP_Model

# Add a random binary column to serve as the decoy feature
df_adv = df.copy()
df_adv["unrelated_column"] = np.random.RandomState(SEED).choice([0, 1], size=df_adv.shape[0])

X_adv_df = df_adv.drop(columns=["class"])
y_adv = df_adv["class"].values

cat_cols_adv = X_adv_df.select_dtypes(include=["category", "object"]).columns.tolist() + ["unrelated_column"]
num_cols_adv = [c for c in X_adv_df.columns if c not in cat_cols_adv]

X_tr_adv_raw, X_te_adv_raw, y_tr_adv, y_te_adv = train_test_split(
    X_adv_df, y_adv, test_size=0.2, random_state=SEED, stratify=y_adv
)

prep_adv = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols_adv),
        ("cat", OrdinalEncoder(), cat_cols_adv),
    ]
)

X_tr_adv = prep_adv.fit_transform(X_tr_adv_raw)
X_te_adv = prep_adv.transform(X_te_adv_raw)

feat_names_adv = num_cols_adv + list(prep_adv.named_transformers_["cat"].get_feature_names_out(cat_cols_adv))
cat_indcs_adv = [feat_names_adv.index(c) for c in cat_cols_adv]

sex_idx = feat_names_adv.index("sex")
unrel_idx = feat_names_adv.index("unrelated_column")


def one_hot_encode(y_arr: np.ndarray) -> np.ndarray:
    res = np.zeros((len(y_arr), 2))
    res[np.arange(len(y_arr)), y_arr] = 1
    return res


class BiasedModelF:
    """True deployed behavior: predicts strictly based on 'sex'."""
    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([0 if x[sex_idx] == 0 else 1 for x in X])

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return one_hot_encode(self.predict(X))


class InnocuousModelPsi:
    """Decoy behavior shown to LIME/SHAP perturbations: predicts based on 'unrelated_column'."""
    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([0 if x[unrel_idx] > 0 else 1 for x in X])

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return one_hot_encode(self.predict(X))


In [ ]:
# Train adversarial wrapper for LIME
adv_lime = Adversarial_Lime_Model(BiasedModelF(), InnocuousModelPsi()).train(
    X_tr_adv[:500],
    y_tr_adv[:500],
    feature_names=feat_names_adv,
    categorical_features=cat_indcs_adv,
)

# Verify behavioral fidelity on real test data
fidelity = np.mean(adv_lime.predict(X_te_adv) == BiasedModelF().predict(X_te_adv))
print(f"Fidelity of adv_lime to BiasedModelF on real test set: {fidelity * 100:.2f}%")

# Compare LIME explanations on BiasedModelF vs. adv_lime
sample_idx = 5
lime_adv_exp = LimeTabularExplainer(
    X_tr_adv,
    feature_names=adv_lime.get_column_names(),
    discretize_continuous=False,
    categorical_features=cat_indcs_adv,
    random_state=SEED,
)

exp_true_biased = lime_adv_exp.explain_instance(X_te_adv[sample_idx], BiasedModelF().predict_proba, num_features=5)
exp_fooled = lime_adv_exp.explain_instance(X_te_adv[sample_idx], adv_lime.predict_proba, num_features=5)

print("\nLIME top features on TRUE BiasedModelF:", exp_true_biased.as_list())
print("LIME top features on FOOLED adv_lime:   ", exp_fooled.as_list())

In [ ]:
# Visualization of LIME explanation of the biased model.
html = exp_true_biased.as_html()
display(HTML(html))

In [ ]:
# Visualization of LIME explanation of the biased model.
html = exp_fooled.as_html()
display(HTML(html))

In [ ]:
# Security Audit Exercise: How do we catch `adv_lime` red-handed?
# Direct Counterfactual / Flip Test on the suspected sensitive feature (`sex`):
x_probe = X_te_adv[:200].copy()
preds_before = adv_lime.predict(x_probe)

# Flip `sex` (0 -> 1, 1 -> 0) while holding all other features constant
x_probe_flipped = x_probe.copy()
x_probe_flipped[:, sex_idx] = 1.0 - x_probe_flipped[:, sex_idx]
preds_after = adv_lime.predict(x_probe_flipped)

flip_rate = np.mean(preds_before != preds_after)
print(f"Prediction flip rate when toggling ONLY `sex` on 200 real test samples: {flip_rate * 100:.2f}%")

### Comprehension Questions — Part 4
1. **Mechanism of the Attack:** Look at the `Fidelity` score and the two LIME outputs (`BiasedModelF` vs. `adv_lime`). How is `adv_lime` able to agree with `BiasedModelF` on ~95–100% of real test samples while convincing LIME that `unrelated_column` is the most important feature?
2. **Defensive Auditing:** Explain why LIME failed to detect the bias in `adv_lime`, whereas our **Direct Feature Flip Test** (toggling `sex` on in-distribution samples) immediately exposed the discrimination (`flip_rate`). What general lesson does this give security auditors about relying solely on post-hoc feature attributions?

**[STUDENT ANSWER 4.1]:**
> *Write your 2–3 sentence answer here.*

**[STUDENT ANSWER 4.2]:**
> *Write your 2–3 sentence answer here.*

## Fooling SHAP

In [ ]:
# Train the adversarial model
adv_shap = Adversarial_Kernel_SHAP_Model(BiasedModelF(), InnocuousModelPsi()).\
            train(X_tr_adv[:100], y_tr_adv[:100], feature_names=feat_names_adv)

In [ ]:
import shap

# Set the background distribution for the shap explainer using kmeans
# The results get better if we use a lot of samples but at the expense
# of a longer training time
background_distribution = shap.kmeans(X_tr_adv, 100)

# Let's use the shap kernel explainer and grab a point to explain
to_examine = np.random.choice(X_te_adv.shape[0])

# Explain the biased model
biased_kernel_explainer = shap.KernelExplainer(BiasedModelF().predict, background_distribution)
biased_shap_values = biased_kernel_explainer.shap_values(X_te_adv[to_examine:to_examine+1])

# Explain the adversarial model
adv_kernel_explainer = shap.KernelExplainer(adv_shap.predict, background_distribution)
adv_shap_values = adv_kernel_explainer.shap_values(X_te_adv[to_examine:to_examine+1])


In [ ]:
# Plot SHAP values for the biased model.
shap.summary_plot(biased_shap_values,
                  feature_names=feat_names_adv,
                  plot_type="bar")


In [ ]:
# Plot SHAP values for the biased model.
shap.summary_plot(adv_shap_values,
                  feature_names=feat_names_adv,
                  plot_type="bar")


# Part 5 (Optional / GPU Demo): LLM Token Attribution with `LXT`
*Run this section only if you have a CUDA GPU with `meta-llama/Llama-3.2-1B-Instruct` access.*

In [ ]:
if torch.cuda.is_available():
    try:
        import transformers
        from packaging.version import Version
        if Version(transformers.__version__) > Version("5.10"):
            import utils.lxt_shim
        from utils.lxt_vis import visualize_lxt_tokens


        from transformers import AutoTokenizer
        import transformers.models.qwen3.modeling_qwen3 as modeling_qwen3
        from lxt.efficient import monkey_patch
        from lxt.utils import clean_tokens

        monkey_patch(modeling_qwen3, verbose=False)
        #bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
        model_id = "Qwen/Qwen3-4B"

        llm_model = modeling_qwen3.Qwen3ForCausalLM.from_pretrained(
            model_id, device_map="cuda", dtype=torch.bfloat16,)
        llm_tok = AutoTokenizer.from_pretrained(model_id)
        llm_model.train()
        llm_model.gradient_checkpointing_enable()
        for p in llm_model.parameters():
            p.requires_grad = False

        prompt = (
            "Context: The 1922 British Everest expedition pushed the north ridge route up to 8,320 m. "
            "Question: How high did they climb in 1922? Answer: They reached 8,"
        )
        input_ids = llm_tok(prompt, return_tensors="pt", add_special_tokens=True).input_ids.to("cuda")
        input_embeds = llm_model.get_input_embeddings()(input_ids)

        logits = llm_model(inputs_embeds=input_embeds.requires_grad_(), use_cache=False).logits
        max_logit, _ = torch.max(logits[0, -1, :], dim=-1)
        max_logit.backward()

        rel = (input_embeds.grad * input_embeds).float().sum(-1).detach().cpu()[0]
        rel = (rel / (rel.abs().max() + 1e-8)).numpy()
        toks = clean_tokens(llm_tok.convert_ids_to_tokens(input_ids[0]))

        top_idx = np.argsort(-np.abs(rel))[:8]
        print("Top 8 most relevant input tokens for predicting next token:")
        for idx in top_idx:
            print(f"  Token: {toks[idx]!r:15s} | Relevance: {rel[idx]:+.4f}")
    except Exception as e:
        print(f"Skipping Part 5 LLM demo ({e}).")
else:
    print("No CUDA GPU detected — skipping Part 5 optional LLM demo.")


In [ ]:
visualize_lxt_tokens(toks, rel, top_k=10, predicted_token=None)

# Part 6 (Optional / GPU Demo): VLM Attribution with `TAM`
*Run this section only if you have a CUDA GPU with `Qwen/Qwen3.5-2B` access.*

In [ ]:
from io import BytesIO
from PIL import Image
import requests

import torch
from transformers import Qwen3_5ForConditionalGeneration, AutoProcessor


## Load the model

In [ ]:

# bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model_id = "Qwen/Qwen3.5-2B"

vlm_model = Qwen3_5ForConditionalGeneration.from_pretrained(model_id,
                                                            device_map="cuda",
                                                            dtype=torch.bfloat16,
                                                            )

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=False)

## Load the inputs (image + text) and forward pass

In [ ]:
url = "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/CI_Demo/mathv-1327.jpg"

response = requests.get(url, timeout=15)
response.raise_for_status()  # raises an error if the URL fails (404, 403, etc.)

image = Image.open(BytesIO(response.content)).convert("RGB")

text = (
    "Is there a square on this image ? Please answer with yes or no."
)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": text},
        ],
    }
]

# 3. Apply chat template + tokenize + process image tensors in one call
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(vlm_model.device)

In [ ]:
with torch.no_grad():
    outputs = vlm_model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        output_hidden_states=True, # ---> TAM needs hidden states
        return_dict_in_generate=True
    )

In [ ]:
generated_ids = outputs.sequences[0]

# Filter out the input prompt to get only the new tokens
prompt_len = inputs["input_ids"].shape[1]
# Note: In some HF models, generate returns full sequence; others just new tokens.
# You need a safety check here depending on model family.
new_ids = generated_ids[prompt_len:] if generated_ids.shape[0] > prompt_len else generated_ids

decoded_text = processor.batch_decode(
    new_ids.unsqueeze(0),
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]


# Compute logits from last hidden states with vocab classifier for TAM
logits = [vlm_model.lm_head(feats[-1]) for feats in outputs.hidden_states]

In [ ]:
decoded_text

## TAM explanation

### Utils

In [ ]:
import numpy as np

tok = processor.tokenizer

# 1. Dynamically fetch the shifted special tokens
# This automatically grabs 248053, 248054, etc., for Qwen 3.5
vision_start = tok.convert_tokens_to_ids("<|vision_start|>")
vision_end = tok.convert_tokens_to_ids("<|vision_end|>")
im_start = tok.convert_tokens_to_ids("<|im_start|>")
im_end = tok.convert_tokens_to_ids("<|im_end|>")

# 2. Core text tokens (These remain unchanged in the BPE vocab)
newline = 198       # \n
assistant = 77091   # assistant

special_ids = {
    # Image is strictly bounded by <|vision_start|> and <|vision_end|>
    "img_id": [vision_start, vision_end],
    
    # Prompt starts after the image and ends at the assistant header: 
    # <|im_end|>\n<|im_start|>assistant
    "prompt_id": [vision_end, [im_end, newline, im_start, assistant]],
    
    # Answer starts right after the newline following the assistant header
    "answer_id": [[newline, im_start, assistant, newline], -1],
}

# 3. Dynamically calculate the Vision Grid (DO NOT hardcode to 24x24)
image_grid_thw = inputs.get("image_grid_thw")
if image_grid_thw is None:
    raise ValueError("TAM requires image_grid_thw for Qwen models")

# Qwen Vision Transformer applies a 2x2 spatial downsampling.
vision_shape = (
    image_grid_thw[0, 1].item() // 2, 
    image_grid_thw[0, 2].item() // 2
)

vis_inputs = np.array(image.convert("RGB"))

### Generate Explanation

In [ ]:
from explainers.tam import TAM

def compute_tam_attributions(
    generated_ids,
    logits,
    vision_shape,
    special_ids,
    vis_inputs,
    processor,
    target_indices=None,
):
    """
    Compute TAM text and image attributions following the TAMExplainer pattern.
    Returns:
        text_attributions: Tensor of shape (num_targets, seq_len)
        img_attributions: Tensor of shape (num_targets, grid_h, grid_w)
    """
    seq_len_generated = len(logits)
    if target_indices is None:
        indices_to_compute = list(range(seq_len_generated))
    elif isinstance(target_indices, int):
        indices_to_compute = [target_indices]
    else:
        indices_to_compute = list(target_indices)

    indices_to_compute = [idx for idx in indices_to_compute if idx < seq_len_generated]

    img_scores_list = []
    text_attributions = []
    img_attributions = []

    for idx in indices_to_compute:
        result = TAM(
            tokens=generated_ids.cpu().tolist(),
            vision_shape=vision_shape,
            logit_list=logits,
            special_ids=special_ids,
            vision_input=vis_inputs,
            processor=processor,
            save_fn="",
            target_token=idx,
            img_scores_list=img_scores_list,
            eval_only=True,
            return_components=True,
        )
        img_attributions.append(torch.from_numpy(result["img_map_norm"]).float())
        text_attributions.append(torch.from_numpy(result["txt_scores_raw"]).float())

    img_attributions = torch.stack(img_attributions)    # (num_targets, grid_h, grid_w)
    text_attributions = torch.stack(text_attributions)  # (num_targets, input_len)
    return text_attributions, img_attributions


In [ ]:
target_indices = [0] # Change this to the index of the token you want to explain (e.g., 0 for the first generated token)

text_attrs, img_attrs = compute_tam_attributions(
    generated_ids=generated_ids,
    logits=logits,
    vision_shape=vision_shape,
    special_ids=special_ids,
    vis_inputs=vis_inputs,
    processor=processor,
    target_indices=target_indices,
)
print("text_attrs shape:", text_attrs.shape, "| img_attrs shape:", img_attrs.shape)

In [ ]:
from utils.viz_utils import XAIVisualizer

viz = XAIVisualizer(processor)

# Build special_token_ids to exclude <|image_pad|> and chat-template tokens
special_token_ids = list(getattr(processor.tokenizer, "all_special_ids", []))
for tok_str in ["<|image_pad|>", "<|vision_start|>", "<|vision_end|>", "<|vision_pad|>", "<|video_pad|>"]:
    tid = processor.tokenizer.convert_tokens_to_ids(tok_str)
    if isinstance(tid, int) and tid >= 0 and tid not in special_token_ids:
        special_token_ids.append(tid)

target_ids = new_ids.unsqueeze(0)  # shape: (1, gen_len)


### Token visualization

In [ ]:
viz.plot_text_attributions(
    text_attr=text_attrs.float(),
    input_ids=inputs["input_ids"],
    target_ids=target_ids,
    special_token_ids=special_token_ids,
    target_indices=target_indices,
)


### Pixel visualization

In [ ]:
viz.plot_image_attributions(
    img_attr=img_attrs.float(),
    original_image=image,
    target_ids=target_ids,
    image_grid_thw=inputs.get("image_grid_thw", None),
    target_indices=target_indices,
)